### Generator

In [ ]:
from src.rag.components.generator import create_llama_cpp_generator

In [ ]:
prompt = "you are a helpful mortgage advisor"

In [ ]:
from src.rag.schemas.generator_schemas import AnswerModel

In [ ]:
base_url = "http://127.0.0.1:8001"

In [ ]:
llama_cpp_generator = create_llama_cpp_generator(base_url=base_url,)

In [ ]:
llama_cpp_generator._ping_api()

### Reading the question and Paragraphs

In [ ]:
from pathlib import Path

In [ ]:
current_working_directory = Path.cwd()

In [ ]:
data_directory = current_working_directory.joinpath("datasets/halifax_intermediaries/")
data_directory.exists()

In [ ]:
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

In [ ]:
path  = data_directory.joinpath(f"question_and_paragraph_{embedding_model_name.split('/')[-1]}.csv")

In [ ]:
path

In [ ]:
import pandas as pd

In [ ]:
question_paragraph = pd.read_csv(path, index_col=[0, 1])

In [ ]:
question_paragraph = question_paragraph.reset_index(
).loc[question_paragraph.reset_index().Question != 'nan\n']

In [ ]:
question_paragraph = question_paragraph.set_index(['Question', "query_index"])

In [ ]:
question_paragraph.query("Question.str.contains('freehold?')")

In [ ]:
from tqdm import tqdm

generated_answers = {}
for question in tqdm(question_paragraph.index.levels[0], desc="Generating answers"):
    paragraphs = question_paragraph.loc[question].content.tolist()
    template_input = {
        "documents": paragraphs,
        "question": question
    }
    response = llama_cpp_generator.run(template_values=template_input)
    generated_answers[question] = response

In [ ]:
len(generated_answers)

In [ ]:
answers_json = {}
import json
for question, answer in generated_answers.items():
    try:
        answer_json = json.loads(answer)
        answers_json[question] = answer_json
    except Exception as e:
        print(f"Error decoding JSON: {e}")
        print(f"Problematic answer: {answer}")

In [ ]:
generate_answers_df = pd.DataFrame.from_dict(answers_json, orient='index')

In [ ]:
generate_answers_df.head()

In [ ]:
generate_answers_df.answer = generate_answers_df.answer.fillna(generate_answers_df.reason
                                  ).fillna(generate_answers_df.answers).fillna(generate_answers_df.reasoning)

In [ ]:
generate_answers_df["references"] = generate_answers_df.documents_used.fillna(
    generate_answers_df.document_numbers).fillna(generate_answers_df.document_used).fillna(generate_answers_df.used_documents)

In [ ]:
generate_answers_df.loc[generate_answers_df.references.isna(), "references"] = [0]

In [ ]:
generate_answers_df.query("can_answer == False").shape

In [ ]:
question_paragraphs_answers = question_paragraph.reset_index().merge(
    generate_answers_df.reset_index(),
    left_on="Question",
    right_on="index",
    how="inner").set_index(["Question"])

In [ ]:
question_paragraphs_answers = question_paragraphs_answers.reset_index()

In [ ]:
question_paragraphs_answers["chunk_index"] = question_paragraphs_answers.reset_index().groupby("Question").cumcount() + 1

In [ ]:
question_paragraphs_answers.head()

In [ ]:
def filter_group(group):
    references = group['references'].iloc[0]  # All rows in group have same indices
    if not isinstance(references, list):
        references = []
    return group[group['chunk_index'].isin(references)]


result = question_paragraphs_answers.groupby('Question').apply(filter_group).reset_index(drop=True)

In [ ]:
selected_question = result['Question'].drop_duplicates().sample(1).iloc[0]
selected_question

In [ ]:
result.query(f"Question == '{selected_question}'")

In [ ]:
with pd.option_context('display.width', 1000,
                       'display.max_columns', 100,
                       'display.max_colwidth', 1000):
   
    print(result.query(f"Question == '{selected_question}'")[['response', 'content']])

### Hallucination Detection using NLIm

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/nli-deberta-v3-base')

In [ ]:
scores = model.predict([('A man is eating pizza', 'A man eats something')], apply_softmax=True)

In [ ]:
label_mapping = ['contradiction', 'entailment', 'neutral']

In [ ]:
import numpy as np

In [ ]:
print([ "{:0.2f}".format(x) for x in np.round((scores[0] * 100), 2) ])


Tomorrow try RAG function approach.

In [ ]:
sample_values = result.query(f"Question == '{selected_question}'")[["response", "content"]].values

In [ ]:
sample_values.tolist()

In [ ]:
model.predict(sample_values.tolist(),
              apply_softmax=True
              )

In [ ]:
entailment_scores = model.predict(
    result[["response", "content"]].values.tolist(),
    apply_softmax=True,
    batch_size=8
)

In [ ]:
entailment_df = pd.DataFrame(
  (entailment_scores.round(2) * 100),
  columns = label_mapping
)

question_paragraphs_answers = pd.concat([result, entailment_df], axis=1)

In [ ]:
question_paragraphs_answers.

In [ ]:
question_with_references = question_paragraphs_answers.sort_values(by=["Question", "content", "entailment"], ascending=[True, True, False]).drop_duplicates(subset=["Question", "content"])

In [ ]:
question_with_references.sort_values(by=['entailment', "Question"], ascending=[False, False])[
    ["Question", "response", "Answer", "content", "entailment", "contradiction", 'neutral', 'can_answer']].query("contradiction >= 75  & can_answer >= True").shape

In [ ]:

with pd.option_context('display.width', 2000,
                       'display.max_columns', 200,
                       'display.max_colwidth', 2000):
    display(question_with_references.sort_values(by=['entailment', "Question"], ascending=[False, False])[["Question", "response", "Answer", "content", "references", "chunk_index", "entailment", "contradiction", 'neutral', 'can_answer']].query("contradiction >= 75  & can_answer >= True"))

### The entailement bit seems to work half, half, need to try it on a real dataset.

### Trying Vectera Model

from transformers import AutoModelForSequenceClassificationaa

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
vectara_model = AutoModelForSequenceClassification.from_pretrained(
    'vectara/hallucination_evaluation_model', trust_remote_code=True)

In [ ]:
vectara_scores = vectara_model.predict(question_paragraphs_answers[["claim", "content"]].values.tolist(), 
                      )

In [ ]:
question_paragraphs_answers["hallucination_score"] = vectara_scores * 100

In [ ]:
question_paragraphs_answers.query("entailment >= 75 & can_answer == True")